# Merge Sleep-EDF and Kaggle Datasets

This notebook:
1. Aggregates 5,450 EDF epochs into 2 subject-level rows
2. Creates synthetic sleep quality labels based on sleep architecture
3. Generates lifestyle proxy features for EDF subjects
4. Merges with Kaggle dataset (374 + 2 = 376 total subjects)


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Setup paths
ROOT = Path.cwd().parent
REPORTS = ROOT / "reports"
FIGS = ROOT / "figures"

print("Root directory:", ROOT)
print("Reports directory:", REPORTS)


## Step 1: Load Raw EDF Features


In [ ]:
# Load epoch-level EDF features
edf_raw = pd.read_csv(REPORTS / "sleepedf_features.csv")

print(f"EDF raw shape: {edf_raw.shape}")
print(f"\nColumns: {edf_raw.columns.tolist()}")
print(f"\nSubjects: {edf_raw['subject'].unique()}")
print(f"\nStage distribution:\n{edf_raw['stage'].value_counts()}")
edf_raw.head()


## Step 2: Aggregate to Subject-Level Features

Create summary statistics for each subject including:
- Sleep architecture (% time in each stage)
- EEG/EOG/EMG summary statistics
- Sleep efficiency, latency, WASO


In [ ]:
def aggregate_subject_features(df):
    """
    Aggregate epoch-level features to subject-level.
    Each epoch = 30 seconds.
    """
    agg_features = []
    
    for subject in df['subject'].unique():
        subj_data = df[df['subject'] == subject].copy()
        n_epochs = len(subj_data)
        
        # Sleep architecture: percentage of time in each stage
        stage_counts = subj_data['stage'].value_counts()
        stage_pct = (stage_counts / n_epochs * 100).to_dict()
        
        # Total time in hours
        total_time_hours = n_epochs * 30 / 3600  # epochs * 30sec / 3600
        
        # Sleep stages (N1, N2, N3, REM are sleep; W is wake)
        sleep_stages = ['N1', 'N2', 'N3', 'REM']
        n_sleep_epochs = sum([stage_counts.get(s, 0) for s in sleep_stages])
        n_wake_epochs = stage_counts.get('W', 0)
        
        # Sleep efficiency: (total sleep time / total time) * 100
        sleep_efficiency = (n_sleep_epochs / n_epochs * 100) if n_epochs > 0 else 0
        
        # Sleep duration in hours (total sleep time)
        sleep_duration_hours = n_sleep_epochs * 30 / 3600
        
        # Wake after sleep onset (WASO): wake time after first sleep epoch
        first_sleep_idx = subj_data[subj_data['stage'].isin(sleep_stages)].index.min()
        if pd.notna(first_sleep_idx):
            wake_after_sleep = subj_data.loc[first_sleep_idx:][subj_data['stage'] == 'W'].shape[0]
            waso_minutes = wake_after_sleep * 30 / 60
            
            # Sleep latency: time to first sleep stage
            first_sleep_epoch = subj_data[subj_data['stage'].isin(sleep_stages)]['epoch'].min()
            sleep_latency_minutes = first_sleep_epoch * 30 / 60 if pd.notna(first_sleep_epoch) else 0
        else:
            waso_minutes = 0
            sleep_latency_minutes = 0
        
        # EEG/EOG/EMG summary statistics
        eeg_cols = ['eeg_delta_rel', 'eeg_theta_rel', 'eeg_alpha_rel', 'eeg_beta_rel']
        other_cols = ['eog_var', 'emg_rms']
        
        feature_dict = {
            'subject': subject,
            'n_epochs': n_epochs,
            'total_time_hours': total_time_hours,
            'sleep_duration_hours': sleep_duration_hours,
            'sleep_efficiency_pct': sleep_efficiency,
            'sleep_latency_min': sleep_latency_minutes,
            'waso_min': waso_minutes,
            
            # Stage percentages
            'stage_W_pct': stage_pct.get('W', 0),
            'stage_N1_pct': stage_pct.get('N1', 0),
            'stage_N2_pct': stage_pct.get('N2', 0),
            'stage_N3_pct': stage_pct.get('N3', 0),
            'stage_REM_pct': stage_pct.get('REM', 0),
        }
        
        # EEG/EOG/EMG statistics (mean, std, median)
        for col in eeg_cols + other_cols:
            feature_dict[f'{col}_mean'] = subj_data[col].mean()
            feature_dict[f'{col}_std'] = subj_data[col].std()
            feature_dict[f'{col}_median'] = subj_data[col].median()
        
        agg_features.append(feature_dict)
    
    return pd.DataFrame(agg_features)

# Aggregate features
edf_aggregated = aggregate_subject_features(edf_raw)

print(f"Aggregated EDF shape: {edf_aggregated.shape}")
print(f"\nColumns ({len(edf_aggregated.columns)}): {edf_aggregated.columns.tolist()}")
edf_aggregated


## Step 3: Create Synthetic Sleep Quality Labels

Rule-based heuristic:
- **Good sleep (≥7)**: High sleep efficiency (≥85%) + good deep sleep (N3 ≥15%) + good REM (≥20%)
- **Fair sleep (7)**: Moderate efficiency (≥75%) with some deep/REM sleep
- **Poor sleep (<7)**: Otherwise


In [ ]:
def create_synthetic_sleep_quality(row):
    """
    Generate synthetic sleep quality score based on sleep architecture.
    """
    sleep_eff = row['sleep_efficiency_pct']
    n3_pct = row['stage_N3_pct']
    rem_pct = row['stage_REM_pct']
    
    # Excellent sleep
    if sleep_eff >= 85 and n3_pct >= 15 and rem_pct >= 20:
        return 8
    # Good sleep
    elif sleep_eff >= 75 and (n3_pct >= 10 or rem_pct >= 15):
        return 7
    # Fair sleep
    elif sleep_eff >= 65:
        return 6
    # Poor sleep
    else:
        return 5

# Apply labeling function
edf_aggregated['quality_of_sleep'] = edf_aggregated.apply(create_synthetic_sleep_quality, axis=1)

print("Synthetic sleep quality labels:")
print(edf_aggregated[['subject', 'sleep_efficiency_pct', 'stage_N3_pct', 'stage_REM_pct', 'quality_of_sleep']])
print(f"\nLabel distribution: {edf_aggregated['quality_of_sleep'].value_counts().to_dict()}")


## Step 4: Create Lifestyle Proxy Features

Add Kaggle-compatible columns for EDF subjects using:
- `sleep_duration`: Calculated from total sleep time
- Other features: Use dataset median/mode values


In [ ]:
# Load Kaggle dataset to get median/mode values
kaggle = pd.read_csv(REPORTS / "kaggle_clean_winsorized.csv")

print(f"Kaggle shape: {kaggle.shape}")
print(f"\nKaggle columns: {kaggle.columns.tolist()}")

# Calculate median/mode values for proxy features
proxy_values = {
    'age': int(kaggle['age'].median()),
    'gender': kaggle['gender'].mode()[0],
    'occupation': 'Other',  # Use a generic category
    'physical_activity_level': int(kaggle['physical_activity_level'].median()),
    'stress_level': int(kaggle['stress_level'].median()),
    'bmi_category': kaggle['bmi_category'].mode()[0],
    'blood_pressure': kaggle['blood_pressure'].mode()[0],
    'heart_rate': int(kaggle['heart_rate'].median()),
    'daily_steps': int(kaggle['daily_steps'].median()),
    'sleep_disorder': None,  # Unknown
    'sleep_disorder_missing': 1,  # Mark as missing
}

print("\nProxy values for EDF subjects:")
for k, v in proxy_values.items():
    print(f"  {k}: {v}")


In [ ]:
# Add lifestyle proxy features to EDF aggregated data
edf_with_lifestyle = edf_aggregated.copy()

# Use actual sleep duration from EDF data
edf_with_lifestyle['sleep_duration'] = edf_with_lifestyle['sleep_duration_hours']

# Add proxy values for other Kaggle features
for col, value in proxy_values.items():
    edf_with_lifestyle[col] = value

# Create unique person_ids (starting from max Kaggle ID + 1)
max_person_id = kaggle['person_id'].max()
edf_with_lifestyle['person_id'] = range(max_person_id + 1, max_person_id + 1 + len(edf_with_lifestyle))

print(f"EDF with lifestyle features shape: {edf_with_lifestyle.shape}")
print(f"\nSample:")
edf_with_lifestyle[['subject', 'person_id', 'age', 'gender', 'sleep_duration', 'quality_of_sleep']].head()


## Step 5: Save Subject-Level EDF Features


In [ ]:
# Save aggregated EDF features
edf_with_lifestyle.to_csv(REPORTS / "sleepedf_subject_aggregated.csv", index=False)
print(f"✅ Saved: {REPORTS / 'sleepedf_subject_aggregated.csv'}")
print(f"   Shape: {edf_with_lifestyle.shape}")


## Step 6: Merge Kaggle and EDF Datasets

Create unified dataset with:
- 374 Kaggle subjects (lifestyle features + NaN for physiological)
- 2 EDF subjects (lifestyle proxies + physiological features)
- Total: 376 subjects


In [ ]:
# Get Kaggle columns (excluding EDF-specific columns)
kaggle_cols = ['person_id', 'gender', 'age', 'occupation', 'sleep_duration', 'quality_of_sleep',
               'physical_activity_level', 'stress_level', 'bmi_category', 'blood_pressure',
               'heart_rate', 'daily_steps', 'sleep_disorder', 'sleep_disorder_missing']

# Get EDF-specific physiological columns
edf_specific_cols = [col for col in edf_with_lifestyle.columns 
                      if col not in kaggle_cols and col != 'subject']

print(f"Kaggle columns: {len(kaggle_cols)}")
print(f"EDF-specific physiological columns: {len(edf_specific_cols)}")
print(f"\nEDF physiological features (first 10): {edf_specific_cols[:10]}")


In [ ]:
# Add EDF physiological columns to Kaggle data (filled with NaN)
kaggle_extended = kaggle.copy()
for col in edf_specific_cols:
    kaggle_extended[col] = np.nan

# Add 'subject' column to Kaggle (NaN for non-EDF subjects)
kaggle_extended['subject'] = np.nan

print(f"Kaggle extended shape: {kaggle_extended.shape}")
print(f"Columns: {len(kaggle_extended.columns)}")


In [ ]:
# Prepare EDF data with same column order
edf_for_merge = edf_with_lifestyle[kaggle_extended.columns].copy()

print(f"EDF for merge shape: {edf_for_merge.shape}")
print(f"Columns match: {list(edf_for_merge.columns) == list(kaggle_extended.columns)}")


In [ ]:
# Concatenate datasets
merged_data = pd.concat([kaggle_extended, edf_for_merge], axis=0, ignore_index=True)

print(f"\n✅ Merged dataset shape: {merged_data.shape}")
print(f"   Kaggle subjects: {(merged_data['subject'].isna()).sum()}")
print(f"   EDF subjects: {(merged_data['subject'].notna()).sum()}")
print(f"\nMissing data per column (top 10):")
missing_pct = (merged_data.isna().sum() / len(merged_data) * 100).sort_values(ascending=False)
print(missing_pct.head(10))


In [ ]:
# Verify quality_of_sleep distribution
print("Quality of sleep distribution in merged data:")
print(merged_data['quality_of_sleep'].value_counts().sort_index())
print(f"\nBinary split (cutoff=7):")
print(f"  Poor (<7): {(merged_data['quality_of_sleep'] < 7).sum()}")
print(f"  Good (≥7): {(merged_data['quality_of_sleep'] >= 7).sum()}")


## Step 7: Save Merged Dataset


In [ ]:
# Save merged dataset
merged_data.to_csv(REPORTS / "kaggle_edf_merged.csv", index=False)

print(f"\n✅ Saved merged dataset: {REPORTS / 'kaggle_edf_merged.csv'}")
print(f"   Total subjects: {len(merged_data)}")
print(f"   Total features: {len(merged_data.columns)}")
print(f"\nColumn summary:")
print(f"   Lifestyle features: {len(kaggle_cols)}")
print(f"   Physiological features: {len(edf_specific_cols)}")
print(f"   Metadata: subject, person_id")


## Summary

✅ **Data Engineering Complete**

1. Aggregated 5,450 EDF epochs → 2 subject-level records
2. Created synthetic sleep quality labels based on sleep architecture
3. Generated lifestyle proxy features for EDF subjects
4. Merged Kaggle (374) + EDF (2) = 376 total subjects

**Next steps:**
- Update preprocessing pipeline to handle physiological features
- Train hybrid model on merged dataset
- Compare performance against baseline
